## Generate Random Zoom Outs for Train 

In [2]:
import os
import random
from PIL import Image
from collections import defaultdict

def apply_zoom_out(image_path, label_path, zoom_out_factor, output_image_path, output_label_path):
    """
    Apply zoom out augmentation to an image and its labels by adding white borders
    
    Args:
    - image_path: Path to the input image
    - label_path: Path to the input label file
    - zoom_out_factor: Factor by which to zoom out (< 1)
    - output_image_path: Path to save the zoomed out image
    - output_label_path: Path to save the zoomed out labels
    """
    # Load the image
    original_image = Image.open(image_path)
    img_width, img_height = original_image.size

    # Calculate new image dimensions
    new_width = int(img_width / zoom_out_factor)
    new_height = int(img_height / zoom_out_factor)

    # Create a new white background image
    zoomed_out_image = Image.new('RGB', (new_width, new_height), color='white')

    # Calculate positioning to center the original image
    left = (new_width - img_width) // 2
    top = (new_height - img_height) // 2

    # Paste the original image onto the white background
    zoomed_out_image.paste(original_image, (left, top))

    # Adjust bounding box labels
    zoomed_labels = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])

            # Convert normalized coordinates to pixel coordinates of original image
            px_x_center = x_center * img_width
            px_y_center = y_center * img_height
            px_width = width * img_width
            px_height = height * img_height

            # Adjust pixel coordinates to new white background image
            new_px_x_center = px_x_center + left
            new_px_y_center = px_y_center + top

            # Convert back to normalized coordinates of the new image
            new_x_center = new_px_x_center / new_width
            new_y_center = new_px_y_center / new_height
            new_width = px_width / new_width
            new_height = px_height / new_height

            # Strictly enforce valid normalized coordinates
            new_x_center = max(0, min(1, new_x_center))
            new_y_center = max(0, min(1, new_y_center))
            new_width = max(0, min(1, new_width))
            new_height = max(0, min(1, new_height))

            # Ensure the entire bounding box is within image bounds
            if (new_x_center - new_width/2 >= 0 and 
                new_x_center + new_width/2 <= 1 and 
                new_y_center - new_height/2 >= 0 and 
                new_y_center + new_height/2 <= 1):
                # Add the modified label
                zoomed_labels.append(f"{class_id} {new_x_center:.6f} {new_y_center:.6f} {new_width:.6f} {new_height:.6f}")

    # Only save if we have valid labels
    if zoomed_labels:
        # Resize the zoomed out image back to original size (optional, can be removed if you want variable size)
        zoomed_out_image = zoomed_out_image.resize((img_width, img_height), Image.LANCZOS)
        
        # Save the zoomed out image and labels
        zoomed_out_image.save(output_image_path)
        with open(output_label_path, 'w') as f:
            f.writelines('\n'.join(zoomed_labels) + '\n')
        return True
    return False

def balance_dataset(base_path, target_count=175):
    """Generate 175 zoom-out augmentations per class"""
    # Available zoom out factors with more fine-grained progression
    zoom_out_factors = [round(0.88 - x * 0.04, 2) for x in range(7)]  # [0.88, 0.84, 0.80, 0.76, 0.72, 0.68, 0.64]
    
    # Define the output directory for augmented data
    output_base_path = "F:/ds-final-ben-zoomout/train"
    os.makedirs(output_base_path, exist_ok=True)

    # Process each class directory
    class_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d)) and d.startswith('class')]
    
    for class_dir in class_dirs:
        class_path = os.path.join(base_path, class_dir)
        images_dir = os.path.join(class_path, 'images')
        labels_dir = os.path.join(class_path, 'labels')
        
        # Ensure directories exist
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)
        
        # Get list of original images (without zoom annotations)
        original_images = [f for f in os.listdir(images_dir) 
                         if f.endswith(('.jpg', '.jpeg', '.png')) and '(z' not in f]
        
        print(f"\nProcessing {class_dir}:")
        print(f"Original images: {len(original_images)}")
        
        # Ensure at least 175 images are generated per class
        print(f"Generating {target_count} augmented images...")
        
        # Keep track of used zoom out factors for each image
        used_zoom_factors = defaultdict(set)
        
        augmentations_created = 0
        attempts = 0
        max_attempts = target_count * 40  # Increased attempts to account for potential rejections
        
        while augmentations_created < target_count and attempts < max_attempts:
            attempts += 1
            
            # Get a random original image
            image_file = random.choice(original_images)
            
            # Get available zoom out factors for this image
            available_factors = [f for f in zoom_out_factors 
                               if f not in used_zoom_factors[image_file]]
            
            if not available_factors:
                continue  # Skip if no available zoom out factors for this image
                
            # Choose random zoom out factor from available ones
            zoom_out_factor = random.choice(available_factors)
            used_zoom_factors[image_file].add(zoom_out_factor)
            
            # Setup paths for output in the new directory
            label_file = os.path.splitext(image_file)[0] + '.txt'
            image_path = os.path.join(images_dir, image_file)
            label_path = os.path.join(labels_dir, label_file)
            
            # Verify label file exists
            if not os.path.exists(label_path):
                print(f"Warning: Label file not found for {image_file}")
                continue
            
            # Create output filenames with zoom out factor
            base_name = os.path.splitext(image_file)[0]
            ext = os.path.splitext(image_file)[1]
            new_image_name = f"{base_name}(z{zoom_out_factor}){ext}"
            new_label_name = f"{base_name}(z{zoom_out_factor}).txt"
            
            output_image_path = os.path.join(output_base_path, class_dir, 'images', new_image_name)
            output_label_path = os.path.join(output_base_path, class_dir, 'labels', new_label_name)
            
            # Ensure the output directories exist
            os.makedirs(os.path.dirname(output_image_path), exist_ok=True)
            os.makedirs(os.path.dirname(output_label_path), exist_ok=True)
            
            # Apply zoom out augmentation
            if apply_zoom_out(image_path, label_path, zoom_out_factor, output_image_path, output_label_path):
                augmentations_created += 1
                if augmentations_created % 10 == 0:
                    print(f"Generated {augmentations_created}/{target_count} augmented images")
            
        # Print statistics
        print(f"\nCompleted augmentation for {class_dir}")
        print(f"Total augmented images created: {augmentations_created}")
        print("\nZoom out factors used per image:")
        for img, factors in used_zoom_factors.items():
            if factors:
                print(f"  {img}: {sorted(factors)}")

if __name__ == "__main__":
    # Path to your dataset directory containing class0, class1, etc.
    dataset_path = "F:/ds-final-ben/train"  # Adjust this to your dataset path
    
    print("Starting Generating balancing...")
    balance_dataset(dataset_path)
    print("\nDataset Generating completed!")


Starting Generating balancing...


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'F:/ds-final-ben/train'

## Generate Random Zoom Outs for Val

In [ ]:
import os
import random
from PIL import Image
from collections import defaultdict

def apply_zoom_out(image_path, label_path, zoom_out_factor, output_image_path, output_label_path):
    """
    Apply zoom out augmentation to an image and its labels by adding white borders
    
    Args:
    - image_path: Path to the input image
    - label_path: Path to the input label file
    - zoom_out_factor: Factor by which to zoom out (< 1)
    - output_image_path: Path to save the zoomed out image
    - output_label_path: Path to save the zoomed out labels
    """
    # Load the image
    original_image = Image.open(image_path)
    img_width, img_height = original_image.size

    # Calculate new image dimensions
    new_width = int(img_width / zoom_out_factor)
    new_height = int(img_height / zoom_out_factor)

    # Create a new white background image
    zoomed_out_image = Image.new('RGB', (new_width, new_height), color='white')

    # Calculate positioning to center the original image
    left = (new_width - img_width) // 2
    top = (new_height - img_height) // 2

    # Paste the original image onto the white background
    zoomed_out_image.paste(original_image, (left, top))

    # Adjust bounding box labels
    zoomed_labels = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])

            # Convert normalized coordinates to pixel coordinates of original image
            px_x_center = x_center * img_width
            px_y_center = y_center * img_height
            px_width = width * img_width
            px_height = height * img_height

            # Adjust pixel coordinates to new white background image
            new_px_x_center = px_x_center + left
            new_px_y_center = px_y_center + top

            # Convert back to normalized coordinates of the new image
            new_x_center = new_px_x_center / new_width
            new_y_center = new_px_y_center / new_height
            new_width = px_width / new_width
            new_height = px_height / new_height

            # Strictly enforce valid normalized coordinates
            new_x_center = max(0, min(1, new_x_center))
            new_y_center = max(0, min(1, new_y_center))
            new_width = max(0, min(1, new_width))
            new_height = max(0, min(1, new_height))

            # Ensure the entire bounding box is within image bounds
            if (new_x_center - new_width/2 >= 0 and 
                new_x_center + new_width/2 <= 1 and 
                new_y_center - new_height/2 >= 0 and 
                new_y_center + new_height/2 <= 1):
                # Add the modified label
                zoomed_labels.append(f"{class_id} {new_x_center:.6f} {new_y_center:.6f} {new_width:.6f} {new_height:.6f}")

    # Only save if we have valid labels
    if zoomed_labels:
        # Resize the zoomed out image back to original size (optional, can be removed if you want variable size)
        zoomed_out_image = zoomed_out_image.resize((img_width, img_height), Image.LANCZOS)
        
        # Save the zoomed out image and labels
        zoomed_out_image.save(output_image_path)
        with open(output_label_path, 'w') as f:
            f.writelines('\n'.join(zoomed_labels) + '\n')
        return True
    return False

def balance_dataset(base_path, target_count=50):
    # Available zoom out factors with more fine-grained progression
    zoom_out_factors = [round(0.88 - x * 0.04, 2) for x in range(7)]  # [0.88, 0.84, 0.80, 0.76, 0.72, 0.68, 0.64]
    
    # Define the output directory for augmented data
    output_base_path = "F:/ds-final-ben-zoomout/val"
    os.makedirs(output_base_path, exist_ok=True)

    # Process each class directory
    class_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d)) and d.startswith('class')]
    
    for class_dir in class_dirs:
        class_path = os.path.join(base_path, class_dir)
        images_dir = os.path.join(class_path, 'images')
        labels_dir = os.path.join(class_path, 'labels')
        
        # Ensure directories exist
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)
        
        # Get list of original images (without zoom annotations)
        original_images = [f for f in os.listdir(images_dir) 
                         if f.endswith(('.jpg', '.jpeg', '.png')) and '(z' not in f]
        
        print(f"\nProcessing {class_dir}:")
        print(f"Original images: {len(original_images)}")
        
        # Ensure at least 175 images are generated per class
        print(f"Generating {target_count} augmented images...")
        
        # Keep track of used zoom out factors for each image
        used_zoom_factors = defaultdict(set)
        
        augmentations_created = 0
        attempts = 0
        max_attempts = target_count * 40  # Increased attempts to account for potential rejections
        
        while augmentations_created < target_count and attempts < max_attempts:
            attempts += 1
            
            # Get a random original image
            image_file = random.choice(original_images)
            
            # Get available zoom out factors for this image
            available_factors = [f for f in zoom_out_factors 
                               if f not in used_zoom_factors[image_file]]
            
            if not available_factors:
                continue  # Skip if no available zoom out factors for this image
                
            # Choose random zoom out factor from available ones
            zoom_out_factor = random.choice(available_factors)
            used_zoom_factors[image_file].add(zoom_out_factor)
            
            # Setup paths for output in the new directory
            label_file = os.path.splitext(image_file)[0] + '.txt'
            image_path = os.path.join(images_dir, image_file)
            label_path = os.path.join(labels_dir, label_file)
            
            # Verify label file exists
            if not os.path.exists(label_path):
                print(f"Warning: Label file not found for {image_file}")
                continue
            
            # Create output filenames with zoom out factor
            base_name = os.path.splitext(image_file)[0]
            ext = os.path.splitext(image_file)[1]
            new_image_name = f"{base_name}(z{zoom_out_factor}){ext}"
            new_label_name = f"{base_name}(z{zoom_out_factor}).txt"
            
            output_image_path = os.path.join(output_base_path, class_dir, 'images', new_image_name)
            output_label_path = os.path.join(output_base_path, class_dir, 'labels', new_label_name)
            
            # Ensure the output directories exist
            os.makedirs(os.path.dirname(output_image_path), exist_ok=True)
            os.makedirs(os.path.dirname(output_label_path), exist_ok=True)
            
            # Apply zoom out augmentation
            if apply_zoom_out(image_path, label_path, zoom_out_factor, output_image_path, output_label_path):
                augmentations_created += 1
                if augmentations_created % 10 == 0:
                    print(f"Generated {augmentations_created}/{target_count} augmented images")
            
        # Print statistics
        print(f"\nCompleted augmentation for {class_dir}")
        print(f"Total augmented images created: {augmentations_created}")
        print("\nZoom out factors used per image:")
        for img, factors in used_zoom_factors.items():
            if factors:
                print(f"  {img}: {sorted(factors)}")

if __name__ == "__main__":
    # Path to your dataset directory containing class0, class1, etc.
    dataset_path = "F:/ds-final-ben/val"  # Adjust this to your dataset path
    
    print("Starting Generating balancing...")
    balance_dataset(dataset_path)
    print("\nDataset Generating completed!")


## Generated zoom out count for Train and Val

In [ ]:
## Generated zoom out count for Train and Val 
import os
import matplotlib.pyplot as plt

# Path to your training dataset
train_dir = "F:/ds-final-ben-zoomout/train"

# Count images per class
class_counts = {}

for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name, "images")
    if os.path.isdir(class_path):
        class_counts[class_name] = len([f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))])

# Sort the counts for better visualization
class_counts = dict(sorted(class_counts.items(), key=lambda x: x[0]))

# Plot the chart
plt.figure(figsize=(12, 6))
plt.bar(class_counts.keys(), class_counts.values(), color="skyblue")
plt.xlabel("Class", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.title("Training Zoom out data", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.tight_layout()
plt.show()

import os
import matplotlib.pyplot as plt

# Path to your training dataset
train_dir = "F:/ds-final-ben-zoomout/val"

# Count images per class
class_counts = {}

for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name, "images")
    if os.path.isdir(class_path):
        class_counts[class_name] = len([f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))])

# Sort the counts for better visualization
class_counts = dict(sorted(class_counts.items(), key=lambda x: x[0]))

# Plot the chart
plt.figure(figsize=(12, 6))
plt.bar(class_counts.keys(), class_counts.values(), color="skyblue")
plt.xlabel("Class", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.title("Validation Zoom out data", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.tight_layout()
plt.show()


## Generate Random Zoom In for training set

In [ ]:
## Generate Random Zoom for training set
import os
import random
from PIL import Image
from collections import defaultdict

def apply_zoom(image_path, label_path, zoom_factor, output_image_path, output_label_path):
    """
    Apply zoom augmentation to an image and its labels with improved bounding box adjustment
    
    Args:
    - image_path: Path to the input image
    - label_path: Path to the input label file
    - zoom_factor: Factor by which to zoom (> 1)
    - output_image_path: Path to save the zoomed image
    - output_label_path: Path to save the zoomed labels
    """
    # Load the image
    image = Image.open(image_path)
    img_width, img_height = image.size

    # Calculate crop dimensions for zoom
    crop_width = img_width / zoom_factor
    crop_height = img_height / zoom_factor
    left = (img_width - crop_width) / 2
    top = (img_height - crop_height) / 2
    right = left + crop_width
    bottom = top + crop_height

    # Crop and resize the image back to original dimensions
    cropped_image = image.crop((left, top, right, bottom))
    zoomed_image = cropped_image.resize((img_width, img_height), Image.LANCZOS)

    # Adjust bounding box labels
    zoomed_labels = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])

            # Convert normalized coordinates to pixel coordinates
            px_x_center = x_center * img_width
            px_y_center = y_center * img_height
            px_width = width * img_width
            px_height = height * img_height

            # Adjust pixel coordinates based on crop
            px_x_center = (px_x_center - left) * zoom_factor
            px_y_center = (px_y_center - top) * zoom_factor

            # Adjust pixel width and height
            px_width *= zoom_factor
            px_height *= zoom_factor

            # Convert back to normalized coordinates
            new_x_center = px_x_center / img_width
            new_y_center = px_y_center / img_height
            new_width = px_width / img_width
            new_height = px_height / img_height

            # Strictly enforce valid normalized coordinates
            new_x_center = max(0, min(1, new_x_center))
            new_y_center = max(0, min(1, new_y_center))
            new_width = max(0, min(1, new_width))
            new_height = max(0, min(1, new_height))

            # Ensure the entire bounding box is within image bounds
            if (new_x_center - new_width/2 >= 0 and 
                new_x_center + new_width/2 <= 1 and 
                new_y_center - new_height/2 >= 0 and 
                new_y_center + new_height/2 <= 1):
                # Add the modified label
                zoomed_labels.append(f"{class_id} {new_x_center:.6f} {new_y_center:.6f} {new_width:.6f} {new_height:.6f}")

    # Only save if we have valid labels
    if zoomed_labels:
        # Save the zoomed image and labels
        zoomed_image.save(output_image_path)
        with open(output_label_path, 'w') as f:
            f.writelines('\n'.join(zoomed_labels) + '\n')
        return True
    return False

def balance_dataset(base_path, target_count=350):
    """Balance all classes to have target_count images using zoom augmentation"""
    # Available zoom factors with more fine-grained progression
    zoom_factors = [round(x * 0.02 + 1.04, 2) for x in range(7)]  # [1.04, 1.06, 1.08, 1.10, 1.12, 1.14, 1.16]
    
    # Process each class directory
    class_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d)) and d.startswith('class')]
    
    for class_dir in class_dirs:
        class_path = os.path.join(base_path, class_dir)
        images_dir = os.path.join(class_path, 'images')
        labels_dir = os.path.join(class_path, 'labels')
        
        # Ensure directories exist
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)
        
        # Get list of original images (without zoom annotations)
        original_images = [f for f in os.listdir(images_dir) 
                         if f.endswith(('.jpg', '.jpeg', '.png')) and '(z' not in f]
        
        print(f"\nProcessing {class_dir}:")
        print(f"Original images: {len(original_images)}")
        
        current_count = len([f for f in os.listdir(images_dir) 
                           if f.endswith(('.jpg', '.jpeg', '.png'))])
        num_needed = max(0, target_count - current_count)
        
        print(f"Current total images: {current_count}")
        print(f"Additional images needed: {num_needed}")
        
        if num_needed == 0:
            print("No augmentation needed for this class")
            continue

        # Keep track of used zoom factors for each image
        used_zoom_factors = defaultdict(set)
        
        # Calculate maximum augmentations possible
        max_possible = len(original_images) * len(zoom_factors)
        if max_possible < num_needed:
            print(f"Warning: Can only generate {max_possible} unique augmentations "
                  f"({len(original_images)} images × {len(zoom_factors)} zoom factors)")
            num_needed = max_possible
        
        augmentations_created = 0
        attempts = 0
        max_attempts = num_needed * 20  # Increased attempts to account for potential rejections
        
        while augmentations_created < num_needed and attempts < max_attempts:
            attempts += 1
            
            # Get a random original image
            image_file = random.choice(original_images)
            
            # Get available zoom factors for this image
            available_factors = [f for f in zoom_factors 
                               if f not in used_zoom_factors[image_file]]
            
            if not available_factors:
                continue  # Skip if no available zoom factors for this image
                
            # Choose random zoom factor from available ones
            zoom_factor = random.choice(available_factors)
            used_zoom_factors[image_file].add(zoom_factor)
            
            # Setup paths
            label_file = os.path.splitext(image_file)[0] + '.txt'
            image_path = os.path.join(images_dir, image_file)
            label_path = os.path.join(labels_dir, label_file)
            
            # Verify label file exists
            if not os.path.exists(label_path):
                print(f"Warning: Label file not found for {image_file}")
                continue
            
            # Create output filenames with zoom factor
            base_name = os.path.splitext(image_file)[0]
            ext = os.path.splitext(image_file)[1]
            new_image_name = f"{base_name}(z{zoom_factor}){ext}"
            new_label_name = f"{base_name}(z{zoom_factor}).txt"
            
            output_image_path = os.path.join(images_dir, new_image_name)
            output_label_path = os.path.join(labels_dir, new_label_name)
            
            # Apply zoom augmentation
            if apply_zoom(image_path, label_path, zoom_factor, output_image_path, output_label_path):
                augmentations_created += 1
                if augmentations_created % 10 == 0:
                    print(f"Generated {augmentations_created}/{num_needed} augmented images")
            
        # Print statistics
        print(f"\nCompleted augmentation for {class_dir}")
        print(f"Total augmented images created: {augmentations_created}")
        print("\nZoom factors used per image:")
        for img, factors in used_zoom_factors.items():
            if factors:
                print(f"  {img}: {sorted(factors)}")

if __name__ == "__main__":
    # Path to your dataset directory containing class0, class1, etc.
    dataset_path = "F:/ds-final-ben/train"  # Adjust this to your dataset path
    
    print("Starting dataset balancing...")
    balance_dataset(dataset_path)
    print("\nDataset balancing completed!")

## Generate Random Zoom In for validation set

In [ ]:
## Generate Random Zoom for validation set
import os
import random
from PIL import Image
from collections import defaultdict

def apply_zoom(image_path, label_path, zoom_factor, output_image_path, output_label_path):
    """
    Apply zoom augmentation to an image and its labels with improved bounding box adjustment
    
    Args:
    - image_path: Path to the input image
    - label_path: Path to the input label file
    - zoom_factor: Factor by which to zoom (> 1)
    - output_image_path: Path to save the zoomed image
    - output_label_path: Path to save the zoomed labels
    """
    # Load the image
    image = Image.open(image_path)
    img_width, img_height = image.size

    # Calculate crop dimensions for zoom
    crop_width = img_width / zoom_factor
    crop_height = img_height / zoom_factor
    left = (img_width - crop_width) / 2
    top = (img_height - crop_height) / 2
    right = left + crop_width
    bottom = top + crop_height

    # Crop and resize the image back to original dimensions
    cropped_image = image.crop((left, top, right, bottom))
    zoomed_image = cropped_image.resize((img_width, img_height), Image.LANCZOS)

    # Adjust bounding box labels
    zoomed_labels = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])

            # Convert normalized coordinates to pixel coordinates
            px_x_center = x_center * img_width
            px_y_center = y_center * img_height
            px_width = width * img_width
            px_height = height * img_height

            # Adjust pixel coordinates based on crop
            px_x_center = (px_x_center - left) * zoom_factor
            px_y_center = (px_y_center - top) * zoom_factor

            # Adjust pixel width and height
            px_width *= zoom_factor
            px_height *= zoom_factor

            # Convert back to normalized coordinates
            new_x_center = px_x_center / img_width
            new_y_center = px_y_center / img_height
            new_width = px_width / img_width
            new_height = px_height / img_height

            # Strictly enforce valid normalized coordinates
            new_x_center = max(0, min(1, new_x_center))
            new_y_center = max(0, min(1, new_y_center))
            new_width = max(0, min(1, new_width))
            new_height = max(0, min(1, new_height))

            # Ensure the entire bounding box is within image bounds
            if (new_x_center - new_width/2 >= 0 and 
                new_x_center + new_width/2 <= 1 and 
                new_y_center - new_height/2 >= 0 and 
                new_y_center + new_height/2 <= 1):
                # Add the modified label
                zoomed_labels.append(f"{class_id} {new_x_center:.6f} {new_y_center:.6f} {new_width:.6f} {new_height:.6f}")

    # Only save if we have valid labels
    if zoomed_labels:
        # Save the zoomed image and labels
        zoomed_image.save(output_image_path)
        with open(output_label_path, 'w') as f:
            f.writelines('\n'.join(zoomed_labels) + '\n')
        return True
    return False

def balance_dataset(base_path, target_count=100):
    """Balance all classes to have target_count images using zoom augmentation"""
    # Available zoom factors with more fine-grained progression
    zoom_factors = [round(x * 0.02 + 1.04, 2) for x in range(7)]  # [1.04, 1.06, 1.08, 1.10, 1.12, 1.14, 1.16]
    
    # Process each class directory
    class_dirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d)) and d.startswith('class')]
    
    for class_dir in class_dirs:
        class_path = os.path.join(base_path, class_dir)
        images_dir = os.path.join(class_path, 'images')
        labels_dir = os.path.join(class_path, 'labels')
        
        # Ensure directories exist
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)
        
        # Get list of original images (without zoom annotations)
        original_images = [f for f in os.listdir(images_dir) 
                         if f.endswith(('.jpg', '.jpeg', '.png')) and '(z' not in f]
        
        print(f"\nProcessing {class_dir}:")
        print(f"Original images: {len(original_images)}")
        
        current_count = len([f for f in os.listdir(images_dir) 
                           if f.endswith(('.jpg', '.jpeg', '.png'))])
        num_needed = max(0, target_count - current_count)
        
        print(f"Current total images: {current_count}")
        print(f"Additional images needed: {num_needed}")
        
        if num_needed == 0:
            print("No augmentation needed for this class")
            continue

        # Keep track of used zoom factors for each image
        used_zoom_factors = defaultdict(set)
        
        # Calculate maximum augmentations possible
        max_possible = len(original_images) * len(zoom_factors)
        if max_possible < num_needed:
            print(f"Warning: Can only generate {max_possible} unique augmentations "
                  f"({len(original_images)} images × {len(zoom_factors)} zoom factors)")
            num_needed = max_possible
        
        augmentations_created = 0
        attempts = 0
        max_attempts = num_needed * 20  # Increased attempts to account for potential rejections
        
        while augmentations_created < num_needed and attempts < max_attempts:
            attempts += 1
            
            # Get a random original image
            image_file = random.choice(original_images)
            
            # Get available zoom factors for this image
            available_factors = [f for f in zoom_factors 
                               if f not in used_zoom_factors[image_file]]
            
            if not available_factors:
                continue  # Skip if no available zoom factors for this image
                
            # Choose random zoom factor from available ones
            zoom_factor = random.choice(available_factors)
            used_zoom_factors[image_file].add(zoom_factor)
            
            # Setup paths
            label_file = os.path.splitext(image_file)[0] + '.txt'
            image_path = os.path.join(images_dir, image_file)
            label_path = os.path.join(labels_dir, label_file)
            
            # Verify label file exists
            if not os.path.exists(label_path):
                print(f"Warning: Label file not found for {image_file}")
                continue
            
            # Create output filenames with zoom factor
            base_name = os.path.splitext(image_file)[0]
            ext = os.path.splitext(image_file)[1]
            new_image_name = f"{base_name}(z{zoom_factor}){ext}"
            new_label_name = f"{base_name}(z{zoom_factor}).txt"
            
            output_image_path = os.path.join(images_dir, new_image_name)
            output_label_path = os.path.join(labels_dir, new_label_name)
            
            # Apply zoom augmentation
            if apply_zoom(image_path, label_path, zoom_factor, output_image_path, output_label_path):
                augmentations_created += 1
                if augmentations_created % 10 == 0:
                    print(f"Generated {augmentations_created}/{num_needed} augmented images")
            
        # Print statistics
        print(f"\nCompleted augmentation for {class_dir}")
        print(f"Total augmented images created: {augmentations_created}")
        print("\nZoom factors used per image:")
        for img, factors in used_zoom_factors.items():
            if factors:
                print(f"  {img}: {sorted(factors)}")

if __name__ == "__main__":
    # Path to your dataset directory containing class0, class1, etc.
    dataset_path = "F:/ds-final-ben/val"  # Adjust this to your dataset path
    
    print("Starting dataset balancing...")
    balance_dataset(dataset_path)
    print("\nDataset balancing completed!")